In [0]:
# Step 1: Read Sales data from Bronze

sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/sales/"
    )

display(sales_df)

In [0]:
# Step 2A: Create UDF function for snake_case

import re

def to_snake_case(column_name):
    column_name = re.sub(
        r'(?<!^)(?=[A-Z])',
        '_',
        column_name
    )
    column_name = column_name.replace(" ", "_")
    column_name = column_name.replace("-", "_")
    return column_name.lower()


# Step 2: Convert Sales column headers to snake_case

for column in sales_df.columns:
    sales_df = sales_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

print(sales_df.columns)

In [0]:
# Step 3: Check Sales columns

print(sales_df.columns)

In [0]:
# Step 4A: Reload Sales data from Bronze

sales_df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/sales/"
    )

In [0]:
# Step 4B: Convert Sales column headers to snake_case

for column in sales_df.columns:
    sales_df = sales_df.withColumnRenamed(
        column,
        to_snake_case(column)
    )

print(sales_df.columns)

In [0]:
# Step 4C: Format Sales date columns to yyyy-MM-dd

from pyspark.sql.functions import col, to_date

sales_df = sales_df.withColumn(
    "order_date",
    to_date(
        col("order_date"),
        "yyyy-MM-dd HH:mm:ss"
    )
)

sales_df = sales_df.withColumn(
    "ship_date",
    to_date(
        col("ship_date"),
        "yyyy-MM-dd HH:mm:ss"
    )
)

display(sales_df)

In [0]:
# Step 5: Import DeltaTable

from delta.tables import DeltaTable

# Step 6: Upsert Sales data to Silver

silver_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/customer_sales/"

if DeltaTable.isDeltaTable(spark, silver_path):

    target = DeltaTable.forPath(
        spark,
        silver_path
    )

    target.alias("target") \
        .merge(
            sales_df.alias("source"),
            "target.order_id = source.order_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

    print("Sales data upserted successfully to Silver")

else:

    sales_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(silver_path)

    print("Customer Sales Silver Delta table created successfully")

In [0]:
# Step 7: Verify Customer Sales data in Silver

display(
    spark.read
    .format("delta")
    .load(silver_path)
)